# Imports

In [145]:
import math

import drjit as dr
import mitsuba as mi
import matplotlib.pyplot as plt
import numpy as np

from util import create_and_get_output_path, write_data_json, write_render, write_figure, write_mirror

# Configuration

In [146]:
# Select the variant to use
# Print out to view which you have available
# Need to pick an AD variant for gradient descent
# LLVM for CPU, CUDA for GPU
# Each requires its own installations
# print(mi.variants())
# mi.set_variant('llvm_ad_rgb')
mi.set_variant('cuda_ad_rgb')

# Configuration variables for the simulation
L = 0.1
aperture = 1
half_aperture = aperture / 2
num_mirror1_segments = 1000
spline_control_points = 3
num_2d_vertices = num_mirror1_segments + 1
target_height = (aperture / num_mirror1_segments) * 10
target_half_height = target_height / 2
three_dimensional_thickness = target_half_height
optimization_iterations = 10000
optimization_rays = 10000
optimization_ray_max_y = half_aperture
optimization_ray_min_y = -half_aperture
learning_rate = 0.001
learning_rate_drop_factor = 0.1
learning_rate_drop_percentages = [0.5, 0.75, 0.9]
epsilon = 1e-6
mirror_rough_conductor_alpha = 0.005
test_mean_rays = 11
gradient_clipping_threshold = 0.75
gradient_clipping_norm_threshold = 1
gradient_clipping_drop_factor = 0.5
camera_res_x = 256
camera_res_y = 256
camera_x_pos = -target_height
camera_fov = math.degrees(math.atan(-camera_x_pos / target_half_height))
camera_near_clip = -camera_x_pos / 10
camera_far_clip = -camera_x_pos + 0.1
has_vertex_normals = True
spp=8000
ray_start_x_pos = 1
plot_max_bounce_ray_length = 0.75

# Create an output directory to store info on the run
output_path = create_and_get_output_path('single_mirror_spline')

# Store the configuration
write_data_json({
    'L': L,
    'aperture': aperture,
    'num_mirror1_segments': num_mirror1_segments,
    'spline_control_points': spline_control_points,
    'three_dimensional_thickness': three_dimensional_thickness,
    'target_height': target_height,
    'optimization_iterations': optimization_iterations,
    'optimization_rays': optimization_rays,
    'optimization_ray_max_y': optimization_ray_max_y,
    'optimization_ray_min_y': optimization_ray_min_y,
    'learning_rate': learning_rate,
    'learning_rate_drop_factor': learning_rate_drop_factor,
    'learning_rate_drop_percentages': learning_rate_drop_percentages,
    'epsilon': epsilon,
    'mirror_rough_conductor_alpha': mirror_rough_conductor_alpha,
    'test_mean_rays': test_mean_rays,
    'gradient_clipping_threshold': gradient_clipping_threshold,
    'gradient_clipping_norm_threshold': gradient_clipping_norm_threshold,
    'gradient_clipping_drop_factor': gradient_clipping_drop_factor,
    'camera_res_x': camera_res_x,
    'camera_res_y': camera_res_y,
    'camera_x_pos': camera_x_pos,
    'camera_fov': camera_fov,
    'camera_near_clip': camera_near_clip,
    'camera_far_clip': camera_far_clip,
    'has_vertex_normals': has_vertex_normals,
    'spp': spp,
    'ray_start_x_pos': ray_start_x_pos
}, output_path, 'config')

# Mirror 1 Mesh

In [147]:
# Determine the vertices for the mirror's starting configuration which will just be a flat plane
# Since we're in 3D, have 2 points per y, which is the reason for the doubling
mirror1_y_points = dr.linspace(mi.Float, -half_aperture, half_aperture, num_2d_vertices)
mirror1_y_points = dr.repeat(mirror1_y_points, 2)
mirror1_x_points = dr.full(mi.Float, -L, len(mirror1_y_points))
mirror1_z_points = dr.tile(mi.Float(-three_dimensional_thickness, three_dimensional_thickness), num_2d_vertices)
mirror1_vertices = mi.Point3f(mirror1_x_points, mirror1_y_points, mirror1_z_points)

# Loop through and create 2 triangles per segment
mirror1_face_indices = []
for i in range(num_mirror1_segments):
    i0 = 2 * i                  # lower-left
    i1 = 2 * i + 1              # lower-right
    i2 = 2 * (i + 1)            # upper-left
    i3 = 2 * (i + 1) + 1        # upper-right

    mirror1_face_indices.append([i0, i2, i1])
    mirror1_face_indices.append([i1, i2, i3])

# Flatten out the faces, then reshape them into the expected mitsuba shape
faces_flat = [index for tri in mirror1_face_indices for index in tri]
mirror1_faces = dr.reshape(mi.Vector3u, mi.UInt32(faces_flat), (-1, 3))

# Create the mirror mesh
mirror1_mesh = mi.Mesh(
    'mirror1',
    vertex_count=len(mirror1_y_points),
    face_count=num_mirror1_segments * 2,
    has_vertex_normals=has_vertex_normals,
    has_vertex_texcoords=False,
)

# Add the vertices and faces to the mesh
mirror1_mesh_params = mi.traverse(mirror1_mesh)
mirror1_mesh_params['vertex_positions'] = dr.ravel(mirror1_vertices)
mirror1_mesh_params['faces'] = dr.ravel(mirror1_faces)

# Apply the mesh changes and print info to confirm correct creation
print(mirror1_mesh_params.update())

[(Mesh[
  name = "mirror1",
  bbox = BoundingBox3f[
    min = [-0.1, -0.5, -0.005],
    max = [-0.1, 0.5, 0.005]
  ],
  vertex_count = 2002,
  vertices = [46.9 KiB of vertex data],
  face_count = 2000,
  faces = [23.4 KiB of face data],
  face_normals = 0
], {'vertex_positions', 'faces'})]


# Cameras

In [148]:
# Create a camera to look at the target
# Heavily lifted from the docs: https://mitsuba.readthedocs.io/en/stable/src/inverse_rendering/caustics_optimization.html
target_camera = mi.load_dict({
    'type': 'perspective',
    'near_clip': camera_near_clip,
    'far_clip': camera_far_clip,
    'fov': camera_fov,
    'to_world': mi.ScalarTransform4f().look_at(
        target=[0, 0, 0],
        origin=[camera_x_pos, 0, 0],
        up=[0, 1, 0]
    ),
    'sampler': {
        'type': 'independent',
        'sample_count': 512  # Not really used
    },
    'film': {
        'type': 'hdrfilm',
        'width': camera_res_x,
        'height': camera_res_y,
        'pixel_format': 'luminance',
        'rfilter': {
            # Important: smooth reconstruction filter with a footprint larger than 1 pixel.
            # Use gaussian for a smoother result, but box for less blurring
            'type': 'gaussian'
        }
    },
})

# Create a camera to look at the whole scene
scene_camera = mi.load_dict({
    'type': 'perspective',
    'to_world': mi.ScalarTransform4f().look_at(
        origin=[0, 0, aperture * 2], target=[-L, 0, 0], up=[0, 1, 0]
    ),
})

# Full Scene

In [149]:
# Create the scene
# For the single mirror case, only the mirror is really required
# The light, integrator, and target aren't really used unless you want to render with mitsuba, but left for informational purposes
white_bsdf = mi.load_dict({
        'type': 'diffuse',
        'id': 'white-bsdf',
        'reflectance': { 'type': 'rgb', 'value': (1, 1, 1) },
})

target = mi.load_dict({
    'type': 'rectangle',
    'id': 'target',
    'to_world': mi.ScalarTransform4f.translate([0, 0, 0]) @
                mi.ScalarTransform4f.rotate([0, 1, 0], -90) @ # Rotate to face along X-axis
                mi.ScalarTransform4f.scale([three_dimensional_thickness, target_half_height, 1]),
    'bsdf': white_bsdf,
})

def create_base_scene():
    return {
        'type': 'scene',
        'integrator': {
            'type': 'ptracer',
            'samples_per_pass': spp,
            'max_depth': 4,
            'hide_emitters': True
        },
        'light': {
            'type': 'directional',
            'direction': [-1, 0, 0],
            'irradiance': 1
        }
    }

scene_dict = create_base_scene()
scene_dict['mirror1'] = mirror1_mesh
scene = mi.load_dict(scene_dict)

# Scene Parameters

In [150]:
# Get the scene information
scene_params = mi.traverse(scene)

# Display the scene information
# Not used here as we write custom tracing for a custom loss function, but this is where we can see which parameters we can optimize and are differentiable
print(scene_params)

SceneParameters[
  -----------------------------------------------------------------------------------------
  Name                                  Flags    Type              Parent
  -----------------------------------------------------------------------------------------
  allow_thread_reordering                        bool              Scene
  mirror1.bsdf.reflectance.value        ∂        Float             UniformSpectrum
  mirror1.silhouette_sampling_weight             float             Mesh
  mirror1.faces                                  UInt              Mesh
  mirror1.vertex_positions              ∂, D     Float             Mesh
  mirror1.vertex_normals                ∂, D     Float             Mesh
  mirror1.vertex_texcoords              ∂        Float             Mesh
  light.sampling_weight                          float             DirectionalEmitter
  light.irradiance.value                ∂        Color3f           SRGBReflectanceSpectrum
  light.to_world                

# Optimization

In [ ]:
# Prepare the optimization
target_center = mi.Vector3f(0, 0, 0)
opt = mi.ad.Adam(lr=learning_rate)
opt_x_points = dr.full(mi.Float, -L, spline_control_points)
opt_y_points = dr.linspace(mi.Float, -half_aperture, half_aperture, spline_control_points)
opt['mirror_1_control_points'] = mi.Vector2f(opt_x_points, opt_y_points)

# Prepare the constraints
# We freeze the y of the top-most and bottom-most vertices
mask_x = dr.ones(mi.Float, spline_control_points)
mask_y = dr.ones(mi.Float, spline_control_points)
dr.scatter(mask_x, 0.0, mi.UInt32([spline_control_points // 2]))
dr.scatter(mask_y, 0.0, mi.UInt32([0, spline_control_points // 2, spline_control_points - 1]))

# Prepare random number generation for stochastic rays
range_width = mi.Float(optimization_ray_max_y - optimization_ray_min_y)
range_start = mi.Float(optimization_ray_min_y)
rng = dr.rng()

# Store data for later printing
losses = []

# Prepare the spline parametrization
spline_t = dr.linspace(mi.Float, 0, spline_control_points - 1, num_2d_vertices)
spline_index = dr.clip(mi.UInt32(spline_t), 0, spline_control_points - 2)
spline_segment_t = spline_t - mi.Float(spline_index)

@dr.syntax
def solve_spline_curvature(spline_control_point_coords):
    """
    Solve the system of equations for the curvature of the spline.
    Uses the Thomas Algorithm: https://en.wikipedia.org/wiki/Tridiagonal_matrix_algorithm

    :param spline_control_point_coords: The control point coordinates (either x or y axis).
    """
    c_prime = dr.zeros(mi.Float, spline_control_points)
    d_prime = dr.zeros(mi.Float, spline_control_points)
    curvature = dr.zeros(mi.Float, spline_control_points)

    # Compute the new coefficients by performing a forward sweep
    # Skip the end points as they are not needed (why we skip 0 and skip the last index)
    for j in range(1, spline_control_points - 1):
        # Calculate the right hand side which specifies the bending at this point
        rhs = 6.0 * (spline_control_point_coords[j - 1] - 2.0 * spline_control_point_coords[j] + spline_control_point_coords[j + 1])

        # Hacky boundary logic
        # In the very specific case where there are 3 control points, then both ends need to match, which leads to all curvatures being the same, so 1-4-1 folds into 6
        # For the two end points, we want the curvature to match the curvature of the next point to not flatten out, so the 1-4-1 folds into 5-1 and 1-5 for the respective endpoints when setting the curvatures equal, so the diagonal is 5 in these two cases
        # If we have an internal point, the diagonal value is 4 normally, so leave that as the default case
        if spline_control_points == 3:
            diag = 6.0
        elif j == 1 or j == spline_control_points - 2:
            diag = 5.0
        else:
            diag = 4.0

        # Update the coefficients using the right hand side
        lower_diag = 0.0 if j == 1 else 1.0
        coefficient_denom = diag - lower_diag * c_prime[j - 1]
        c_prime[j] = 1 / coefficient_denom
        d_prime[j] = (rhs - lower_diag * d_prime[j - 1]) / coefficient_denom

    # Perform back substitution to get the actual curvatures
    # Once we got to the last point in the forward pass, we already have its curvature, so we can skip that one
    curvature[spline_control_points - 2] = d_prime[spline_control_points - 2]
    # Start at the last internal point and work our way back
    for j in range(spline_control_points - 3, 0, -1):
        curvature[j] = d_prime[j] - c_prime[j] * curvature[j + 1]

    # Ensure the endpoints match in curvature
    # This makes sure the mirror doesn't flatten out at the ends
    curvature[0] = curvature[1]
    curvature[spline_control_points - 1] = curvature[spline_control_points - 2]

    return curvature

def interpolate_spline(spline_control_point_coords, curvatures):
    """
    Interpolate the spline values based on the provided control point coordinates and the curvature coefficients.
    I saw an excerpt from section 3.3 of Numerical Recipes in C: The Art of Scientific Computing which showed this form.
    Simplified by the fact that the control points are evenly spaced apart in t.

    :param spline_control_point_coords: The control point coordinates (either x or y axis).
    :param curvatures: The curvature coefficients (second derivative).
    """
    # Prepare the points and curvatures that govern the segments
    start_point = dr.gather(mi.Float, spline_control_point_coords, spline_index)
    end_point = dr.gather(mi.Float, spline_control_point_coords, spline_index + 1)
    start_curvature = dr.gather(mi.Float, curvatures, spline_index)
    end_curvature = dr.gather(mi.Float, curvatures, spline_index + 1)

    # Determine the coefficients
    a = 1.0 - spline_segment_t
    b = spline_segment_t
    c = (1.0 / 6.0) * (a ** 3 - a)
    d = (1.0 / 6.0) * (b ** 3 - b)

    # Plug in to the formula to get the spline evaluation
    return (a * start_point) + (b * end_point) + (c * start_curvature) + (d * end_curvature)

def update_scene():
    """
    Mold the mirror vertices into the shape of the spline based on the control points.
    We use a parametrized cubic spline to allow for it to bend over onto itself and not have to worry about duplicate x or y values.
    """
    # Copy the values from the control points from the optimizer
    control_points = opt['mirror_1_control_points']

    # Solve for curvatures
    curvature_x = solve_spline_curvature(control_points.x)
    curvature_y = solve_spline_curvature(control_points.y)

    # Interpolate the spline for the mirror segments
    curve_x = interpolate_spline(control_points.x, curvature_x)
    curve_y = interpolate_spline(control_points.y, curvature_y)

    # Double the vertices for thickness (Z-axis)
    new_x = dr.repeat(curve_x, 2)
    new_y = dr.repeat(curve_y, 2)
    new_z = dr.tile(mi.Float(-three_dimensional_thickness, three_dimensional_thickness), num_2d_vertices)

    # Reshape and modify the scene
    scene_params['mirror1.vertex_positions'] = dr.ravel(mi.Point3f(new_x, new_y, new_z))

    # Propagate changes through the scene (e.g. rebuild BVH)
    scene_params.update()

# The main optimization loop
# Run the update of the scene one time to make sure everything is loaded in correctly
update_scene()
int_learning_rate_drop_percentages = [int(p * optimization_iterations) for p in learning_rate_drop_percentages]
for i in range(optimization_iterations):
    # Reduce the learning rate if we are at certain iteration thresholds
    if int_learning_rate_drop_percentages and i >= int_learning_rate_drop_percentages[0]:
        int_learning_rate_drop_percentages.pop(0)
        opt.lr *= learning_rate_drop_factor
        gradient_clipping_threshold *= gradient_clipping_drop_factor
        gradient_clipping_norm_threshold *= gradient_clipping_drop_factor

    # Generate rays to use for the computation of loss
    # Rays are linearly-spaced across a broad area which will more than cover the first mirror
    # Alternatively, can randomly generate the origins so as not to fit to particular rays and to not allow the optimization to "get used to" the ray positions
    # Max is not differentiable, so we just overshoot to avoid DrJIT issues
    # Linearly-spaced rays
    # ray_ys = dr.linspace(mi.Float, optimization_ray_min_y, optimization_ray_max_y, optimization_rays)
    # Randomly-spaced rays
    ray_ys = range_start + range_width * rng.random(mi.Float, optimization_rays)
    ray_dir = mi.Vector3f(-1, 0, 0)
    ray_origins = mi.Point3f(ray_start_x_pos, ray_ys, 0.0)
    rays = mi.Ray3f(o=ray_origins, d=ray_dir)

    # Intersect with the mirror
    si = scene.ray_intersect(rays)
    hit_mask = si.is_valid()

    # Perform a perfect, specular reflection
    # $wo$ is the reflected direction: $wo = wi - 2 * (wi \cdot n) * n$
    wo = rays.d - 2 * dr.dot(rays.d, si.sh_frame.n) * si.sh_frame.n
    reflected_rays = si.spawn_ray(wo)

    # Using a combination of the angle and the distance as the loss
    # Taking the reflected ray and its perpendicular line going through the target to measure that distance as the loss
    # By using a hit mask, we only consider the rays which actually hit the mirror, which removes error from those which missed (which would behave as though they hit the centre)
    line1_dir = mi.Vector2f(wo.x, wo.y)
    line2_dir = mi.Vector2f(wo.y, -wo.x)
    determinant = line1_dir.y * line2_dir.x - line1_dir.x * line2_dir.y
    determinant_t = line2_dir.y * si.p.x - line2_dir.x * si.p.y
    intersection_t = determinant_t / determinant
    hit_point = mi.Point2f(si.p.x, si.p.y)
    intersection = hit_point + intersection_t * line1_dir
    intersection_distance = dr.norm(intersection)
    masked_intersection_distance = dr.select(hit_mask, intersection_distance, 10.0)
    hit_count = dr.count(hit_mask)
    loss = dr.sum(masked_intersection_distance) / dr.maximum(hit_count, 1.0)

    # Perform the gradient descent
    losses.append(loss[0])
    dr.backward(loss)

    # Clip the gradient to avoid super sudden jerks in any direction due to spatial noise
    # Two ideas here are glipping by value or clipping by norm
    # Clipping by value just sets a min and max value
    # Clipping by norm looks at the whole structure and chill it out
    # Both are provided for information
    # Clip by value
    grads = dr.grad(opt['mirror_1_control_points'])
    grads = dr.clip(grads, -gradient_clipping_threshold, gradient_clipping_threshold)
    # Clip by norm
    grads *= dr.minimum(1.0, gradient_clipping_norm_threshold / dr.norm(grads))

    # Update the gradients based on the modifications
    dr.set_grad(opt['mirror_1_control_points'], grads)

    # Take the step
    opt.step()

    # Pin the y values of the top and bottom points
    optimizer_values = opt['mirror_1_control_points']
    optimizer_values.x = dr.select(mask_x == 0, opt_x_points, optimizer_values.x)
    optimizer_values.y = dr.select(mask_y == 0, opt_y_points, optimizer_values.y)
    opt['mirror_1_control_points'] = mi.Vector2f(optimizer_values.x, optimizer_values.y)

    # Get the latest scene version from the optimized values
    update_scene()

    # Log how we're doing in terms of loss
    print(f'Iteration {i + 1:10d}: loss={loss[0]:.20f}')

Iteration          1: loss=0.25095483660697937012
Iteration          2: loss=0.25033354759216308594
Iteration          3: loss=0.24668057262897491455
Iteration          4: loss=0.24919225275516510010
Iteration          5: loss=0.24962970614433288574
Iteration          6: loss=0.24933546781539916992
Iteration          7: loss=0.24853275716304779053
Iteration          8: loss=0.24948868155479431152
Iteration          9: loss=0.24636059999465942383
Iteration         10: loss=0.24551144242286682129
Iteration         11: loss=0.24632598459720611572
Iteration         12: loss=0.24731343984603881836
Iteration         13: loss=0.24682073295116424561
Iteration         14: loss=0.24413727223873138428
Iteration         15: loss=0.24182879924774169922
Iteration         16: loss=0.24414186179637908936
Iteration         17: loss=0.24317327141761779785
Iteration         18: loss=0.24370166659355163574
Iteration         19: loss=0.24239310622215270996
Iteration         20: loss=0.24234451353549957275


# Data Logging

In [ ]:
write_data_json({
    'final_loss': losses[-1] if losses else None,
    'initial_loss': losses[0] if losses else None,
    'min_loss': min(losses) if losses else None,
    'max_loss': max(losses) if losses else None,
    'losses': losses
}, output_path, "results")

# Plot Loss

In [ ]:
loss_fig = plt.figure(figsize=(10, 6))
plt.plot(losses, color='green', linewidth=1)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.yscale('log')
write_figure(loss_fig, output_path, 'loss')
plt.show()

# Final Mirror

In [ ]:
# Extract the optimized data
flat_vertices = scene_params['mirror1.vertex_positions']

# Get the actual values, where we only need half of them as we don't need both the points with the different z values
points_2d = dr.unravel(mi.Point3f, flat_vertices)
final_x = np.array(points_2d.x[::2])
final_y = np.array(points_2d.y[::2])

# Save the mirror data
write_mirror(final_x, final_y, output_path, 'mirror')

# Mirror Plot

In [ ]:
# Create the figure
fig = plt.figure(figsize=(10, 6))

# Draw the mirror by connecting the vertices
plt.plot(final_x, final_y, 'b-', lw=3, label='Optimized Mirror')

# Draw the target plane and centre
plt.axvline(x=0, color='red', linestyle='--', label='Target Plane (x=0)')
plt.scatter([0], [target_center.y], color='red', s=50, zorder=5, label='Target Centre')

# Draw the ideal parabola
mid_x = final_x[len(final_x) // 2]
parabola_y = np.linspace(-half_aperture, half_aperture, num_mirror1_segments)
parabola_x = parabola_y**2 / (4 * abs(mid_x)) + mid_x
plt.plot(parabola_x, parabola_y, 'y--', lw=1, label="Ideal parabola")

# Labels and formatting
plt.title('2D Single-Mirror Optimization')
plt.grid(True, linestyle=':', alpha=0.6)
plt.gca().set_aspect('equal', adjustable='box')
fig.legend(loc='outside right')

# Save
write_figure(fig, output_path, 'optimized_mirror')

# Display
plt.show()

# Mean Ray Test

In [ ]:
# Compute some mean rays
# TODO: Clean this up. A lot.
test_ray_ys = dr.linspace(mi.Float, final_y[0] + epsilon, final_y[-1] - epsilon, test_mean_rays)
test_ray_dir = mi.Vector3f(-1, 0, 0)
test_ray_origins = mi.Point3f(ray_start_x_pos, test_ray_ys, 0.0)
test_rays = mi.Ray3f(o=test_ray_origins, d=test_ray_dir)
test_si = scene.ray_intersect(test_rays)
test_wo = test_rays.d - 2 * dr.dot(test_rays.d, test_si.sh_frame.n) * test_si.sh_frame.n
cross_points = -test_si.p.x / test_wo.x
cross_points_y = test_si.p.y + cross_points * test_wo.y
test_x0 = np.array(test_ray_origins)[0]
test_y0 = np.array(test_ray_origins)[1]
test_x1 = np.array(test_si.p.x)
test_y1 = np.array(test_si.p.y)
plot_t = np.where((cross_points > 0) & (cross_points < plot_max_bounce_ray_length), cross_points, plot_max_bounce_ray_length)
test_x2 = np.array(test_si.p.x) + np.array(test_wo.x) * plot_t
test_y2 = np.array(test_si.p.y) + np.array(test_wo.y) * plot_t
mean_x = [list(x) for x in zip(*[test_x0, test_x1, test_x2])]
mean_y = [list(y) for y in zip(*[test_y0, test_y1, test_y2])]

# Create the figure
fig = plt.figure(figsize=(10, 6))

# Draw the mirror by connecting the vertices
plt.plot(final_x, final_y, 'b-', lw=3, label='Optimized Mirror')

# Draw the target plane and centre
plt.axvline(x=0, color='red', linestyle='--', label='Target Plane (x=0)')
plt.scatter([0], [target_center.y], color='red', s=50, zorder=5, label='Target Centre')

# Labels and formatting
plt.title('2D Single-Mirror Optimization with Mean Rays')
plt.grid(True, linestyle=':', alpha=0.6)
plt.gca().set_aspect('equal', adjustable='box')

# Plot the mean rays
for i in range(len(mean_x)):
    plt.plot(mean_x[i], mean_y[i], 'y-', lw=1, label='Mean Rays')

# Save
write_figure(fig, output_path, 'optimized_mirror_with_mean_rays')

# Display
plt.show()

# Final Scene

In [ ]:
mirror1_mesh.set_bsdf(white_bsdf)
final_scene_image = mi.render(scene, sensor=scene_camera, spp=spp)
final_scene_max_val = dr.max(final_scene_image)
final_scene_image = final_scene_image / final_scene_max_val
write_render(final_scene_image, output_path, "scene_render")
mi.util.convert_to_bitmap(final_scene_image)

# Optimized Mirror Mesh

In [ ]:
# Search for the mirror in the scene
mirror_shape = None
for shape in scene.shapes():
    if "mirror1" in shape.id():
        mirror_shape = shape
        break

# Create the mirror mesh from the optimized data in the scene
optimized_mirror1_mesh = mi.Mesh(
    'optimized_mirror1',
    vertex_count=len(mirror1_y_points),
    face_count=num_mirror1_segments * 2,
    has_vertex_normals=has_vertex_normals,
    has_vertex_texcoords=False,
)

# Add the vertices and faces to the mesh
optimized_mirror1_mesh_params = mi.traverse(optimized_mirror1_mesh)
optimized_mirror1_mesh_params['vertex_positions'] = mirror_shape.vertex_positions_buffer()
optimized_mirror1_mesh_params['faces'] = mirror_shape.faces_buffer()

# Apply the mesh changes and print info to confirm correct creation
print(optimized_mirror1_mesh_params.update())

# Full Scene with Target

In [ ]:
final_scene_dict = create_base_scene()
final_scene_dict['optimized_mirror1'] = optimized_mirror1_mesh
final_scene_dict['target'] = target
final_scene = mi.load_dict(final_scene_dict)

# Final Target

In [ ]:
# Make the mirror a true mirror
# This only matters if you want to render the scene with mitsuba, which isn't strictly necessary as we plot with matplotlib later, but leaving it in for interest
# The rough-conductor mirror is also provided for debugging, but with particle tracing we should be able to use an ideal mirror as we are forward rendering
ideal_mirror_bsdf = mi.load_dict({
    'type': 'conductor',
    'material': 'none'
})
mirror_bsdf = mi.load_dict({
    'type': 'roughconductor',
    'alpha': mirror_rough_conductor_alpha,
})
optimized_mirror1_mesh.set_bsdf(ideal_mirror_bsdf)
# optimized_mirror1_mesh.set_bsdf(mirror_bsdf)

final_target_image = mi.render(final_scene, sensor=target_camera, spp=spp)
final_target_max_val = dr.max(final_target_image)
final_target_image = final_target_image / final_target_max_val
write_render(final_target_image, output_path, "target_render")
mi.util.convert_to_bitmap(final_target_image)